# Hindi AI Content Quality — Training Notebook

Downloads an open Hindi corpus, creates provisional weak labels, trains four classifiers, evaluates them and saves artifacts. **These labels are heuristic, not expert ground truth.** Run from the project root.

In [ ]:
from pathlib import Path
import sys, subprocess, json
ROOT=Path.cwd()
if not (ROOT/'src').exists() and (ROOT.parent/'src').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
print(ROOT)


In [ ]:
# If needed, run once: %pip install -r requirements.txt


In [ ]:
data=ROOT/'data/hindi_corpus.csv'
if not data.exists(): subprocess.run([sys.executable,str(ROOT/'download_data.py'),'--limit','5000'],check=True,cwd=ROOT)


In [ ]:
import pandas as pd
from src.quality import DIMS,normalize_text,weak_labels
df=pd.read_csv(data).dropna(subset=['text']); df['text']=df.text.map(normalize_text); df=df[df.text.str.len()>=20].drop_duplicates('text').reset_index(drop=True)
print('Rows:',len(df)); display(df.head())


In [ ]:
labels=pd.DataFrame([weak_labels(t) for t in df.text]); train_df=pd.concat([df[['text']],labels],axis=1); display(train_df.head()); display(labels.describe())


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report,f1_score
import joblib
if len(train_df)<50: raise ValueError('Need at least 50 usable rows; increase download limit.')
Xtr,Xte,ytr,yte=train_test_split(train_df.text,train_df[DIMS],test_size=.2,random_state=42)
model=Pipeline([('tfidf',TfidfVectorizer(analyzer='char',ngram_range=(2,5),max_features=100000,min_df=2,sublinear_tf=True)),('classifier',MultiOutputClassifier(LogisticRegression(max_iter=300,class_weight='balanced')))])
model.fit(Xtr,ytr); pred=model.predict(Xte); print('train/test',len(Xtr),len(Xte))


In [ ]:
metrics={}
for i,d in enumerate(DIMS):
 metrics[d]={'macro_f1':float(f1_score(yte.iloc[:,i],pred[:,i],average='macro',zero_division=0)),'report':classification_report(yte.iloc[:,i],pred[:,i],zero_division=0,output_dict=True)}
 print(d,metrics[d]['macro_f1']); print(classification_report(yte.iloc[:,i],pred[:,i],zero_division=0))


In [ ]:
out=ROOT/'models'; out.mkdir(exist_ok=True); joblib.dump(model,out/'quality_model.joblib')
(out/'metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
(out/'model_info.json').write_text(json.dumps({'model_type':'TF-IDF char n-grams + MultiOutput Logistic Regression','training_rows':len(Xtr),'test_rows':len(Xte),'label_source':'weak heuristics, not human annotation'},indent=2),encoding='utf-8')
print('Saved model artifacts:',out)


In [ ]:
sample='यह एक उदाहरण वाक्य है।'; dict(zip(DIMS,[int(v) for v in model.predict([sample])[0]]))


## Improve this baseline
Use `docs/annotation_guidelines.md` to collect expert/human labels. For semantic relevance, keep the original prompt. For cultural appropriateness, use diverse annotators and report agreement and failure cases.